In [3]:
import requests
import pandas as pd
import time
import urllib3

# SSL xəbərdarlıqlarını gizlətmək üçün
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Sizin göndərdiyiniz şəkildəki Authorization (Bearer) tokenini bura tam şəkildə yapışdırın:
TOKEN = "eyJhbGciOiJodHRwOi8vd3d3LnczLm9yZy8yMDAxLzA0L3htbGRzaWctbW9yZSNobWFjLXNoYTUxMiIsInR5cCI6IkpXVCJ9.eyJuYW1laWQiOiIyMiIsInByZWZlcnJlZF91c2VybmFtZSI6InNhaGlsX21lbnNpbW92X3NlZnRlciIsIm5hbWUiOiJTYWhpbCBNyZluc2ltb3YiLCJyb2xlcyI6WyJtyZlsdW1hdF9raXRhYsOnYWxhcsSxX2JheMSxxZ8iLCJzZXJ0aWZpa2F0bGFyYV9iYXjEscWfIiwidHBvX2JheMSxxZ8iLCJiyZl5YW5uYW3JmWzJmXLJmV9iYXjEscWfIiwibGlzZW56aXlhbGFyYV9iYXjEscWfIiwixZ_JmXhzbMmZcsmZX2JheMSxxZ8iLCJxdXLEn3VfYXZhZGFubMSxcV9iYXjEscWfIiwicmV5ZXN0cmxhcmFfYmF4xLHFnyJdLCJuYmYiOjE3ODI4MjY0MDksImV4cCI6MTc4MzY5MDQwOSwiaXNzIjoiZHBvcmVnaXN0cnlAcmVnaXN0cnkuY29tIiwiYXVkIjoiZHBvcmVnaXN0cnlAcmVnaXN0cnkuY29tIn0.1UMd2l_4_80IDBcHIIcc9KcQ_u0vhoNSC6eRHrGVrP8uxlbji9MWwqDlrXh2HNwLiYgZnFStaVvbsWGALy02YA" # <--- Bura şəkildəki kodun hamısını yazın

# API Ünvanları
MAIN_API_URL = "https://api-dpo.fhn.gov.az/api/v1/Reports/get-list"
DETAIL_API_URL = "https://api-dpo.fhn.gov.az/api/v1/Reports/get-detail"

# Sorğu başlıqları (Headers)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
}

# POST sorğusu ilə bütün 10,711 datanı birdən istəyirik (Payload)
payload = {
    "pageIndex": 1,
    "pageSize": 11000,  # Bütün datanı tək səfərdə çəkmək üçün
    "searchText": ""
}

print("FHN Portalı ilə əlaqə yaradılır, ana liste endirilir...")

# Sayt POST metodu işlətdiyi üçün requests.post istifadə edirik
response = requests.post(MAIN_API_URL, json=payload, headers=headers, verify=False)

if response.status_code == 200:
    data = response.json()
    items = data.get("items", data.get("data", data))
    
    # Əgər data birbaşa siyahı deyilsə, gələn struktura uyğunlaşdırırıq
    if isinstance(items, dict) and "list" in items:
        items = items["list"]
    elif isinstance(items, dict) and "data" in items:
        items = items["data"]
        
    all_objects = []
    total = len(items) if isinstance(items, list) else 0
    
    if total == 0:
        print("Siyahı boş gəldi. API strukturunu yoxlayın.")
        exit()
        
    print(f"Uğurlu! Toplam {total} obyekt tapıldı. İndi daxili reyestr detalları çəkilir...")

    for index, item in enumerate(items, 1):
        # Strukturdan asılı olaraq İd nömrəsini tapırıq
        obj_id = item.get("id") or item.get("Id")
        
        reyestr_no = "Yoxdur"
        bitme_tarixi = "Yoxdur"
        
        # Hər bir obyektin daxilinə (Göz işarəsinə) sorğu atırıq
        if obj_id:
            try:
                # Göz işarəsinin daxili API-na müraciət (GET sorğusu ilə)
                detail_resp = requests.get(f"{DETAIL_API_URL}?id={obj_id}", headers=headers, verify=False)
                if detail_resp.status_code == 200:
                    detail_data = detail_resp.json()
                    
                    # Daxildəki reyestr cədvəlinin parametrləri (Dövlət reyestr qeydiyyat çıxarışı)
                    # Saytın daxili strukturuna uyğun açarları yoxlayırıq
                    reyestr_no = detail_data.get("registryNo") or detail_data.get("certificateNo", "Tapılmadı")
                    bitme_tarixi = detail_data.get("endDate") or detail_data.get("expireDate", "Tapılmadı")
            except Exception:
                pass
        
        # Məlumatları təmizləyib siyahıya yığırıq
        all_objects.append({
            "Şəxsin növü": item.get("personType", item.get("personTypeName", "")),
            "Hüquqi(fiziki) şəxsin adı": item.get("legalName") or item.get("companyName") or item.get("name", ""),
            "Vöen": item.get("voen") or item.get("vuen", ""),
            "Telefon nömrəsi": item.get("phone") or item.get("phoneNumber", ""),
            "Tabeçiliyi": item.get("dependency") or item.get("tabecilik", ""),
            "Obyektin adı": item.get("objectName", ""),
            "Obyektin kodu": item.get("objectCode", ""),
            "Obyektin koordinatları": f"{item.get('latitude', '')} {item.get('longitude', '')}",
            "Obyektin təyinatı": item.get("objectDestination", item.get("objectType", "")),
            "Obyektin yerləşdiyi ünvan": item.get("address", ""),
            "Reyestr №": reyestr_no,
            "Çıxarışın bitmə tarixi": bitme_tarixi
        })
        
        # Hər 100 sətirdə bir ekrana gedişatı yazdırırıq
        if index % 100 == 0 or index == total:
            print(f"Tərəqqi: {index}/{total} obyekt tamamlandı...")
            # Saytın bizi bloklamaması üçün çox qısa fasilə
            time.sleep(0.2)

    # Excel formatında yadda saxlayırıq
    df = pd.DataFrame(all_objects)
    df.to_excel("fhn_reyestr_tam_siyahı.xlsx", index=False)
    print("\nƏla! Bütün məlumatlar 'fhn_reyestr_tam_siyahı.xlsx' faylına yazıldı.")

else:
    print(f"Xəta baş verdi! Ana siyahı çəkilə bilmədi. Status Kodu: {response.status_code}")
    print("Məsləhət: Şəkildəki Tokenin vaxtı bitmiş (expire) ola bilər. Səhifəni yeniləyib yeni tokeni kopyalayın.")

FHN Portalı ilə əlaqə yaradılır, ana liste endirilir...
Xəta baş verdi! Ana siyahı çəkilə bilmədi. Status Kodu: 500
Məsləhət: Şəkildəki Tokenin vaxtı bitmiş (expire) ola bilər. Səhifəni yeniləyib yeni tokeni kopyalayın.
